In [1]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.svm import SVR
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import warnings

In [2]:
df=pd.read_csv("Data/Tennis_Usable.csv")

In [3]:
train_size = int(len(df) * 0.8)

train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

X_train=train_df.drop(columns=['Winner_Encoded'])
X_test=test_df.drop(columns=['Winner_Encoded'])
y_train=train_df['Winner_Encoded']
y_test=test_df['Winner_Encoded']

In [4]:
X_train.shape

(54138, 12)

In [11]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

scaler=StandardScaler()
encoder=OneHotEncoder()

num_features = X_train.select_dtypes(exclude="object").columns
cat_features = X_train.select_dtypes(include="object").columns

preprocessor=ColumnTransformer(
    [
        ("StandardScaler",scaler,num_features),
        ("OneHotEncoder",encoder,cat_features)
    ]
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_22768\1915936303.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train.select_dtypes(include="object").columns


In [12]:
cat_features

Index(['Court', 'Surface', 'Round', 'Season'], dtype='str')

In [13]:
num_features

Index(['Best of', 'Player1_H2H_WinPct', 'RecentWinPct_1', 'RecentWinPct_2',
       'Surface_WinPct_1', 'Surface_WinPct_2', 'Higher_Rank_Player',
       'Abs_Rank_Diff'],
      dtype='str')

In [14]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [15]:
X_train

array([[-0.4839669 , -0.00405027, -0.03053035, ...,  0.        ,
         0.        ,  1.        ],
       [-0.4839669 , -0.00405027, -0.03053035, ...,  0.        ,
         0.        ,  1.        ],
       [-0.4839669 , -0.00405027, -0.03053035, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [-0.4839669 , -0.00405027,  2.02108814, ...,  1.        ,
         0.        ,  0.        ],
       [-0.4839669 ,  0.5883825 ,  0.37979335, ...,  1.        ,
         0.        ,  0.        ],
       [-0.4839669 , -0.00405027,  1.20044074, ...,  1.        ,
         0.        ,  0.        ]], shape=(54138, 25))

In [16]:
X_train_dataframe=pd.DataFrame(X_train)
X_train_dataframe.head()

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,-0.483967,-0.00405,-0.03053,-0.026318,-0.046194,-0.041013,-0.998892,-0.226952,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.483967,-0.00405,-0.03053,-0.026318,-0.046194,-0.041013,-0.998892,1.199760,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.483967,-0.00405,-0.03053,-0.026318,-0.046194,-0.041013,-0.998892,-0.191284,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.483967,-0.00405,-0.03053,-0.026318,-0.046194,-0.041013,1.001109,-0.521211,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.483967,-0.00405,-0.03053,-0.026318,-0.046194,-0.041013,-0.998892,0.602324,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [17]:
X_test_dataframe=pd.DataFrame(X_test)
X_test_dataframe.head()

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,-0.483967,-0.00405,-1.261501,0.383914,-1.938270,0.633895,-0.998892,-0.289371,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-0.483967,-0.00405,-0.440854,0.383914,0.663334,-0.498821,1.001109,-0.432042,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.483967,-0.00405,-0.440854,-0.436550,-0.244202,-0.444961,1.001109,-0.280454,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,-0.483967,-0.00405,-1.261501,-1.257013,-0.257994,-0.253086,-0.998892,-0.066447,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-0.483967,-0.00405,-2.082149,-1.257013,-0.312267,-2.868647,1.001109,0.334816,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [18]:
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted)
    recall = recall_score(true, predicted) 
    f1 = f1_score(true, predicted)
    return accuracy, precision, recall, f1

In [70]:
models = {
    "Logistic Regression Classifier": LogisticRegression(),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    ##"Decision Tree Classifier": DecisionTreeClassifier(),
    ##"Random Forest Classifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(), 
    "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_accuracy, model_train_precison, model_train_recall, model_train_f1_score = evaluate_model(y_train, y_train_pred)

    model_test_accuracy, model_test_precison, model_test_recall, model_test_f1_score = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy Score: {:.4f}".format(model_train_accuracy))
    print("- Precision Score: {:.4f}".format(model_train_precison))
    print("- Recall Score: {:.4f}".format(model_train_recall))
    print("- F1 Score: {:.4f}".format(model_train_f1_score))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Accuracy Score: {:.4f}".format(model_test_accuracy))
    print("- Precision Score: {:.4f}".format(model_test_precison))
    print("- Recall Score: {:.4f}".format(model_test_recall))
    print("- F1 Score: {:.4f}".format(model_test_f1_score))
    
    print('='*35)
    print('\n')

Logistic Regression Classifier
Model performance for Training set
- Accuracy Score: 0.6503
- Precision Score: 0.6503
- Recall Score: 0.6508
- F1 Score: 0.6506
----------------------------------
Model performance for Test set
- Accuracy Score: 0.6579
- Precision Score: 0.6573
- Recall Score: 0.6561
- F1 Score: 0.6567


K-Neighbors Classifier
Model performance for Training set
- Accuracy Score: 0.7411
- Precision Score: 0.7425
- Recall Score: 0.7387
- F1 Score: 0.7406
----------------------------------
Model performance for Test set
- Accuracy Score: 0.6126
- Precision Score: 0.6112
- Recall Score: 0.6137
- F1 Score: 0.6124


XGBClassifier
Model performance for Training set
- Accuracy Score: 0.7319
- Precision Score: 0.7307
- Recall Score: 0.7351
- F1 Score: 0.7329
----------------------------------
Model performance for Test set
- Accuracy Score: 0.6601
- Precision Score: 0.6591
- Recall Score: 0.6598
- F1 Score: 0.6594


CatBoosting Classifier
Model performance for Training set
- Accur

In [71]:
params = {

    "Logistic Regression Classifier": {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["liblinear", "lbfgs"],
        "penalty": ["l2"]
    },

    "K-Neighbors Classifier": {
        "n_neighbors": [3, 5, 7, 9, 11, 15],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },

    # "Decision Tree Classifier": {
    #     "criterion": ["gini", "entropy", "log_loss"],
    #     "max_depth": [None, 5, 10, 20],
    #     "min_samples_split": [2, 5, 10],
    #     "min_samples_leaf": [1, 2, 5]
    # },

    # "Random Forest Classifier": {
    #     "n_estimators": [100, 200, 300],
    #     "max_depth": [None, 10, 20],
    #     "min_samples_split": [2, 5, 10],
    #     "min_samples_leaf": [1, 2, 5],
    #     "max_features": ["sqrt", "log2"]
    # },

    "XGBClassifier": {
        "learning_rate": [0.01, 0.05, 0.1],
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },

    "CatBoosting Classifier": {
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "iterations": [100, 300, 500]
    },

    "AdaBoost Classifier": {
        "n_estimators": [50, 100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]
    }
}

In [ ]:
from sklearn.model_selection import GridSearchCV
for i in range(len(models)):
    model = list(models.values())[i]
    para=params[list(models.keys())[i]]

    gs = GridSearchCV(model,para,cv=3)
    gs.fit(X_train,y_train)
    print(list(models.keys())[i])
    print("-----------------------------")
    print(gs.best_params_)

In [19]:
models = {
    "Logistic Regression Classifier": LogisticRegression(C=0.01, penalty='l2', solver='lbfgs'),
    "K-Neighbors Classifier": KNeighborsClassifier(metric= 'manhattan', n_neighbors= 15, weights= 'uniform'),
    ##"Decision Tree Classifier": DecisionTreeClassifier(),
    ##"Random Forest Classifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(colsample_bytree= 1.0, learning_rate= 0.05, max_depth= 3, n_estimators= 300, subsample= 0.8), 
    "CatBoosting Classifier": CatBoostClassifier(verbose=False, depth= 4, iterations= 300, learning_rate= 0.1),
    "AdaBoost Classifier": AdaBoostClassifier(learning_rate= 1.0, n_estimators= 300)
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_accuracy, model_train_precison, model_train_recall, model_train_f1_score = evaluate_model(y_train, y_train_pred)

    model_test_accuracy, model_test_precison, model_test_recall, model_test_f1_score = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy Score: {:.4f}".format(model_train_accuracy))
    print("- Precision Score: {:.4f}".format(model_train_precison))
    print("- Recall Score: {:.4f}".format(model_train_recall))
    print("- F1 Score: {:.4f}".format(model_train_f1_score))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Accuracy Score: {:.4f}".format(model_test_accuracy))
    print("- Precision Score: {:.4f}".format(model_test_precison))
    print("- Recall Score: {:.4f}".format(model_test_recall))
    print("- F1 Score: {:.4f}".format(model_test_f1_score))
    
    print('='*35)
    print('\n')

c:\Users\Hp\OneDrive\Desktop\MACHINE LEARNING\Tennis Rankings Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Logistic Regression Classifier
Model performance for Training set
- Accuracy Score: 0.6541
- Precision Score: 0.6543
- Recall Score: 0.6536
- F1 Score: 0.6539
----------------------------------
Model performance for Test set
- Accuracy Score: 0.6400
- Precision Score: 0.6403
- Recall Score: 0.6385
- F1 Score: 0.6394


K-Neighbors Classifier
Model performance for Training set
- Accuracy Score: 0.6923
- Precision Score: 0.6917
- Recall Score: 0.6939
- F1 Score: 0.6928
----------------------------------
Model performance for Test set
- Accuracy Score: 0.6200
- Precision Score: 0.6189
- Recall Score: 0.6243
- F1 Score: 0.6216


XGBClassifier
Model performance for Training set
- Accuracy Score: 0.6744
- Precision Score: 0.6738
- Recall Score: 0.6760
- F1 Score: 0.6749
----------------------------------
Model performance for Test set
- Accuracy Score: 0.6477
- Precision Score: 0.6491
- Recall Score: 0.6429
- F1 Score: 0.6460


CatBoosting Classifier
Model performance for Training set
- Accur